# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hafsa-SE/flyrank-assignment/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

**Lane 2: Refresh / Content Opportunity Scoring.**

I'm picking this over the others mainly because the starter data is already shaped for it. There's a `trend_direction`, a `days_since_last_update`, position and demand columns, all sitting right there per page. The numbers in section 3 back this up: over half the eligible pages are currently trending down, and almost a quarter show a "ranks well but aging" pattern. That's not a thin slice — there's real volume to rank and learn from.

It also maps onto a decision I can picture someone actually making: a reviewer with limited time, deciding which page to open first. Lane 1 (pure signal analysis) felt like it could turn into "look at everything, conclude nothing" in week one. Lane 3 (clustering) needs more exploration time before I'd trust naming archetypes honestly. This lane gives me a baseline to beat almost immediately, which feels like the right way to spend the first few weeks. I'll revisit this by week 4 once I've looked at the full warehouse — the starter CSV is 30k rows from 32 clients, so it's a fair first look but not the whole picture.

In [1]:
import os
import pandas as pd

# walk up to the repo root so this works whether the kernel starts in
# work/notebooks/ or the repo root itself
while not os.path.isdir("data/raw") and os.getcwd() != "/":
    os.chdir("..")

assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — check working dir"

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(f"Rows: {len(df):,} | Clients: {df['client_id'].nunique()} | Columns: {df.shape[1]}")
print("trend_direction values:", df['trend_direction'].value_counts().to_dict())

Rows: 30,000 | Clients: 32 | Columns: 44
trend_direction values: {'down': 16262, 'stable': 5962, 'up': 4388, 'new': 2236, 'flat': 1152}


## 2. The question: decision, action, cost of a wrong call

**Decision:** out of thousands of content pages, which one should a reviewer look at first?

**Unit of analysis:** one content page (`content_id`). Not a client, not a day — the action ("go look at this page") happens at the page level.

**Who acts, and what they do:** a content strategist/editor working through a capped list — say the top 20 to 50 pages this week — instead of scanning the full inventory by hand.

**Output:** a ranked queue of pages with a score and a reason code attached, so the reviewer knows *why* a page is near the top, not just that it is.

**Cost of a wrong call, in both directions:**
- Ranking a healthy page too high wastes maybe 20-30 minutes of review time — annoying, cheap, recoverable.
- Missing a page that's genuinely declining is worse: it keeps losing visibility unreviewed, and by the time anyone notices (usually when a client asks why their traffic dropped), the fix comes later and costs more than it would have.

That asymmetry matters for how I'll build this — I'd rather the model lean toward catching real decline than toward being conservative, at least until I have a validation result that says otherwise.

**Why this isn't just "train a model":** FlyRank already has rule-based flags (`health_score` and friends) and they work for the obvious cases. But a page can be fresh *and* declining, or stale *and* low-demand and not worth touching — once you're weighing several correlated signals against each other, a fixed if-then rule starts missing tradeoffs a learned ranking could pick up. That's the actual gap this lane sits in.

In [2]:
# a rough sense of scale: how big is the candidate pool vs. a realistic review capacity?
eligible = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)].drop_duplicates("content_id")
declining = eligible[eligible["trend_direction"] == "down"]

review_capacity_per_week = 50  # a guess at what one reviewer can realistically get through
weeks_to_clear = len(declining) / review_capacity_per_week

print(f"Declining, visible pages in this slice: {len(declining):,}")
print(f"At {review_capacity_per_week}/week, clearing just this backlog would take ~{weeks_to_clear:.0f} weeks")
print("-> ranking matters: nobody is getting through this list unordered.")

Declining, visible pages in this slice: 16,262
At 50/week, clearing just this backlog would take ~325 weeks
-> ranking matters: nobody is getting through this list unordered.


## 3. Quick look at the data (2-3 real numbers)

Loading the starter CSV and checking the numbers that actually justify this lane, rather than taking the guide's word for it.

In [3]:
eligible = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)].drop_duplicates("content_id")

declining = eligible[eligible["trend_direction"] == "down"]
decay_risk = eligible[
    (eligible["avg_position"] > 0)
    & (eligible["avg_position"] <= 10)
    & (eligible["content_age_days"] >= 180)
]

print(f"Eligible pages (visible, at least 90 days old): {len(eligible):,}")
print(f"Currently trending down: {len(declining):,} ({len(declining)/len(eligible)*100:.1f}%)")
print(f"Page-one but aging (avg position <=10, age >=180d): {len(decay_risk):,} ({len(decay_risk)/len(eligible)*100:.1f}%)")
print(f"Median impressions (90d) among eligible pages: {eligible['impressions_90d'].median():.0f}")

Eligible pages (visible, at least 90 days old): 30,000
Currently trending down: 16,262 (54.2%)
Page-one but aging (avg position <=10, age >=180d): 7,076 (23.6%)
Median impressions (90d) among eligible pages: 731


## 4. Careful words: what I can and can't claim

**What this work can say:** observed and directional things — "pages with pattern X were more likely to also show pattern Y in this 90-day window," or "this ranking put more true decliners in the top 50 than the flat rule did." Decision-support language: "review this first," not "this page will recover."

**What it can never say:** that refreshing a page *causes* a recovery (that needs an experiment, not this data), that a model predicted anything about Google's actual ranking algorithm, or that a correlation in a 30k-row anonymized slice generalizes to all 519,606 pages in the warehouse without checking there too.

**The specific trap I'm watching for in this lane:** `trend_direction` and `trend_pct` are how the *label* gets defined (`is_declining_label = trend_direction == "down"`), so they can never also be a feature — that would just be the model reading the answer off the label. Same goes for any product flag like `health_score`: useful as a baseline to beat, never as an input.

In [4]:
# sanity check: confirm the label-source columns aren't accidentally sitting in a features list yet
# (nothing to remove right now since no feature set exists yet — this is just documenting the trap)
label_source_cols = ["trend_direction", "trend_pct"]
print("Columns reserved for the label only, never as features:", label_source_cols)
print("Present in starter data:", [c for c in label_source_cols if c in df.columns])

Columns reserved for the label only, never as features: ['trend_direction', 'trend_pct']
Present in starter data: ['trend_direction', 'trend_pct']


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.